# Cell-cell colocalization for Xenium breast sample

We cannot use the script/fine-tune.py directly because the pipeline is built for the huggingface dataset as the input.     Any external new test dataset should be loaded via "tools.get_embeddings.py" function  
- The breast cancer tumor microenvironment dataset can be downloaded via:  
https://www.10xgenomics.com/products/xenium-in-situ/preview-dataset-human-breast.   
- Publication DOI:   
https://doi.org/10.1038/s41467-023-43458-x

In [1]:
import sys
sys.path.append("/scratch/project_465001820/Spatialformer")
sys.path.append("/scratch/project_465001820/Spatialformer/scripts")
sys.path.append("/scratch/project_465001820/Spatialformer/spatialformer/")
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import networkx as nx
from scipy.spatial import KDTree
from utils.utils import GetPairs, get_adj, split_dataset
from fine_tune import FineTune
import json
from tools import embed_data


/scratch/project_465001820/miniconda3/envs/spatialformer/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading the Xenium count matrix, and cell metadata

In [2]:
adata_breast = sc.read_10x_h5("/scratch/project_465001820/Spatialformer/data/Xenium/outs/cell_feature_matrix.h5")
cell_meta = pd.read_parquet("/scratch/project_465001820/Spatialformer/data/Xenium/outs/cells.parquet")

In [3]:
#add the spatial coordinates to the anndata
adata_breast.obsm["spatial"] = cell_meta[["x_centroid", "y_centroid"]].values
#convert the data type of index into interger
adata_breast.obs.index = adata_breast.obs.index.astype(int)

In [4]:
adata_breast.var["gene_name"] = adata_breast.var.index

In [5]:
adata_breast.obsm["spatial"]

array([[ 847.25991211,  326.19136505],
       [ 826.34199524,  328.03182983],
       [ 848.76691895,  331.74318695],
       ...,
       [7470.15942383, 5119.13205566],
       [7477.73720703, 5128.71281738],
       [7489.3765625 , 5123.19777832]])

Getting the matrix of the cell neighbors

In [6]:
# Getting the asymmetry matrix
sparse_adj, cell_ids = get_adj(sample_dataset = None, anndata = adata_breast, radius = 5, plot = False, sym = True)
### Getting the cell pairs according to the distance
Pairs = GetPairs(sparse_adj, num_workers = 8) #assign the 1:1 negative to the positive

100%|██████████| 4/4 [00:23<00:00,  5.77s/it]

ERROR: There are 161256 nodes not included
The total number of pairs: 
positive pair:4568
negative pair:4568


In [7]:
all_pairs = Pairs.all_pairs
all_labels = Pairs.all_labels

In [8]:
all_labels.shape

(9136,)

Random select 500 cells for evaluating the model

In [9]:
zero_shot_cell_size = 500
selected_pairs, selected_labels = split_dataset(all_pairs, all_labels, n_splits = 0, test_size = None, zero_shot_cell_size = zero_shot_cell_size)
# test_dataloader = data_prepare(sample_name, kfold, num_workers, batch_size, radius=r, test_size = None, zero_shot_cell_size = zero_shot_cell_size, split_mode = "random")

In [10]:
config_path = "/scratch/project_465001820/Spatialformer/config/_config_fine_tune_probe.json"
with open(config_path, 'r') as json_file:
    config = json.load(json_file)
model_ckp_path = "/scratch/project_465001820/Spatialformer/output/checkpoints/step=0096000-train_total_loss=-2.9351-val_total_loss=0.0000.ckpt"
r = 10

Getting the dataloader

In [11]:
test_dataloader = embed_data(adata_breast,
               tissue = "Breast", 
               condition = "Disease",
               method = "gene",
               model_ckp_path = model_ckp_path, 
               batch_size = 4,
               mode = "pair",
               only_loader = True,
               left_cell = selected_pairs[:,0],
               right_cell = selected_pairs[:,1],
               pair_label = selected_labels,
               num_workers = 8,
               reveal_name = False
               )

Spatialformer - INFO - Loading the SpatialFormer model...


[rank: 0] Global seed set to 42


require grad: True
Spatialformer - INFO - Setting the model to the evaluation mode...
Spatialformer - INFO - The model is mapped into cuda
Spatialformer - INFO - Encoding the data into the batch...


Run the model to get the predictions

In [12]:
sample_name = "breast_cancer"
fine_tune_mode = "zero_shot"
all_results = {}
Finetune = FineTune(config, model_ckp_path, sample_name, r, fine_tune_mode, wandb = True, strategy = None)
probe_model = Finetune.probe_model
results = Finetune.test(probe_model, test_dataloader)
#tesing the model
all_results[r] = results[0]

[rank: 0] Global seed set to 42
/scratch/project_465001820/miniconda3/envs/spatialformer/lib/python3.8/site-packages/pytorch_lightning/utilities/parsing.py:269: UserWarning: Attribute 'base_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['base_model'])`.
  rank_zero_warn(


The number of GPUS: 1


2025-06-04 02:18:36,958 - ERROR - Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: junwang666 (junwanggroup). Use `wandb login --relogin` to force relogin


/scratch/project_465001820/miniconda3/envs/spatialformer/lib/python3.8/site-packages/lightning_fabric/plugins/environments/slurm.py:166: PossibleUserWarning: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /scratch/project_465001820/miniconda3/envs/spatialfo ...
  rank_zero_warn(
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


before testing


You are using a CUDA device ('AMD Instinct MI250X') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/scratch/project_465001820/miniconda3/envs/spatialformer/lib/python3.8/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:224: PossibleUserWarning: The dataloader, test_dataloader 0, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 128 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.
  rank_zero_warn(


Testing DataLoader 0: 100%|██████████| 305/305 [00:38<00:00,  7.98it/s]

TypeError: iteration over a 0-d tensor